In [0]:
%run "/Workspace/Users/imsuryya@gmail.com/databricks/source_to_bronze/utils"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

employee_schema = StructType([
    StructField("EmployeeID",   IntegerType(), True),
    StructField("EmployeeName", StringType(),  True),
    StructField("Department",   StringType(),  True),
    StructField("Country",      StringType(),  True),
    StructField("Salary",       IntegerType(), True),
    StructField("Age",          IntegerType(), True)
])

department_schema = StructType([
    StructField("DepartmentID",   StringType(), True),
    StructField("DepartmentName", StringType(), True)
])

country_schema = StructType([
    StructField("CountryCode", StringType(), True),
    StructField("CountryName", StringType(), True)
])

In [0]:
df_employee = (spark.read
                    .format("csv")
                    .schema(employee_schema)
                    .option("header", "true")
                    .load("/Volumes/assingment/bronze/emp/"))

df_department = (spark.read
                      .format("csv")
                      .schema(department_schema)
                      .option("header", "true")
                      .load("/Volumes/assingment/bronze/department/"))

df_country = (spark.read
                   .format("csv")
                   .schema(country_schema)
                   .option("header", "true")
                   .load("/Volumes/assingment/bronze/country/"))

df_employee.display()
df_department.display()
df_country.display()

In [0]:
import re
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def camel_to_snake(name):
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    return re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1).lower()

camel_to_snake_udf = udf(camel_to_snake, StringType())

def rename_columns(df):
    for col_name in df.columns:
        df = df.withColumnRenamed(col_name, camel_to_snake(col_name))
    return df

df_employee   = rename_columns(df_employee)
df_department = rename_columns(df_department)
df_country    = rename_columns(df_country)

df_employee.printSchema()
df_employee.display()

In [0]:
from pyspark.sql.functions import col, trim, upper

df_employee   = df_employee.withColumn("department", trim(upper(col("department")))) \
                            .withColumn("country",    trim(upper(col("country"))))

df_department = df_department.withColumn("department_name", trim(upper(col("department_name"))))

df_country    = df_country.withColumn("country_name", trim(upper(col("country_name"))))

In [0]:
df_joined = (df_employee
                .join(df_department,
                      df_employee["department"] == df_department["department_name"],
                      "left")
                .join(df_country,
                      df_employee["country"] == df_country["country_name"],
                      "left")
                .select(
                    df_employee["employee_id"],
                    df_employee["employee_name"],
                    df_employee["department"],
                    df_employee["country"],
                    df_employee["salary"],
                    df_employee["age"],
                    df_department["department_id"],
                    df_country["country_code"]
                ))

df_joined.display()

In [0]:
df3 = df_joined.dropDuplicates(["employee_id"])

In [0]:
from pyspark.sql.functions import current_date

df3 = df3.withColumn("load_date", current_date())
df3.display()

In [0]:
def write_delta(df, path, mode="overwrite"):
    (df.write
       .format("delta")
       .mode(mode)
       .save(path))

def read_delta(path):
    return spark.read.format("delta").load(path)

def write_csv(df, path, mode="overwrite"):
    (df.write
       .format("csv")
       .mode(mode)
       .option("header", "true")
       .save(path))

In [0]:
write_delta(df_country,    "/Volumes/assingment/silver/country/")
write_delta(df_department, "/Volumes/assingment/silver/department/")
write_delta(df3,           "/Volumes/assingment/silver/emp/")